# Linearity Robustness Experiment

The value attribution method fits a 4-parameter binomial GLM (one weight per
principlist value, no intercept) to recover aggregate value profiles. This
experiment fits an expanded 10-parameter model with pairwise interaction terms,
computes marginal profiles, and checks whether the original rankings are preserved.

In [1]:
DIR = "/Users/payalchandak/Desktop/HVP Preprint/ValueBench"
import os
os.chdir(DIR)

import warnings
from itertools import combinations

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import kendalltau

from src.analysis import load_all_decisions, HUMAN_CONSENSUS
from src.analysis.display_names import get_display_name
from src.analysis.tradeoffs import _build_regression_data
from src.analysis.value_profiles import softmax_profile
from src.response_models.case import VALUE_NAMES

from pathlib import Path
LLM_DIR = Path(DIR) / "data" / "llm_decisions" / "physician_recommendation"

all_decisions = load_all_decisions(llm_dir=LLM_DIR)
decisions = [r for r in all_decisions if any(m.startswith("human/") for m in r.models)]

llm_models = sorted({m for r in decisions for m in r.models if not m.startswith("human/")})
models = llm_models + [HUMAN_CONSENSUS]

print(f"{len(decisions)} cases, {len(llm_models)} LLMs + physician consensus")

50 cases, 12 LLMs + physician consensus


## Step 1 — Fit base and interaction models

**Base model** (4 params): `logit(p_i) = Σ_v  w_v  Δ_{i,v}`

**Interaction model** (10 params): adds `Σ_{v<v’} γ_{v,v’} Δ_{i,v} · Δ_{i,v’}`

Both use a binomial GLM with logit link, no intercept, frequency-weighted by trial count.

In [2]:
INTERACTION_PAIRS = list(combinations(range(len(VALUE_NAMES)), 2))
INTERACTION_LABELS = [f"{VALUE_NAMES[i][:1].upper()}×{VALUE_NAMES[j][:1].upper()}"
                      for i, j in INTERACTION_PAIRS]

print("Base features:", VALUE_NAMES)
print("Interaction features:", INTERACTION_LABELS)


def add_interaction_columns(X_base: np.ndarray) -> np.ndarray:
    """Append 6 pairwise interaction columns to the 4-column base matrix."""
    interactions = np.column_stack([
        X_base[:, i] * X_base[:, j] for i, j in INTERACTION_PAIRS
    ])
    return np.hstack([X_base, interactions])


def fit_glm(X, y, n_trials):
    """Fit binomial GLM (logit, no intercept, freq weights). Returns GLMResults."""
    glm = sm.GLM(
        y, X,
        family=sm.families.Binomial(link=sm.families.links.Logit()),
        freq_weights=n_trials,
    )
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        return glm.fit(disp=False)


def fit_glm_regularized(X, y, n_trials, lam):
    """Fit with L2 penalty on interaction terms only (indices 4–9)."""
    glm = sm.GLM(
        y, X,
        family=sm.families.Binomial(link=sm.families.links.Logit()),
        freq_weights=n_trials,
    )
    n_params = X.shape[1]
    alpha = np.zeros(n_params)
    alpha[4:] = lam
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        return glm.fit_regularized(alpha=alpha, L1_wt=0.0, refit=True)

Base features: ['autonomy', 'beneficence', 'nonmaleficence', 'justice']
Interaction features: ['A×B', 'A×N', 'A×J', 'B×N', 'B×J', 'N×J']


In [3]:
L2_LAMBDAS = [0.01, 0.1, 1.0]

rows = []
for model_id in models:
    display = get_display_name(model_id)

    X_base, y, n_trials = _build_regression_data(decisions, model_id)
    X_full = add_interaction_columns(X_base)

    res_base = fit_glm(X_base, y, n_trials)

    try:
        res_full = fit_glm(X_full, y, n_trials)
        penalty_used = None
    except Exception:
        res_full = None

    if res_full is None or not np.isfinite(res_full.llf):
        for lam in L2_LAMBDAS:
            try:
                res_full = fit_glm_regularized(X_full, y, n_trials, lam)
                if np.isfinite(res_full.llf):
                    penalty_used = lam
                    break
            except Exception:
                continue
        else:
            print(f"  {display}: interaction model failed to converge")
            continue

    base_coefs = dict(zip(VALUE_NAMES, res_base.params))
    full_coefs = dict(zip(VALUE_NAMES, res_full.params[:4]))
    interaction_coefs = dict(zip(INTERACTION_LABELS, res_full.params[4:]))

    rows.append({
        "model_id": model_id,
        "display_name": display,
        "base_coefs": base_coefs,
        "full_coefs": full_coefs,
        "interaction_coefs": interaction_coefs,
        "penalty": penalty_used,
    })

    tag = f"  (L2={penalty_used})" if penalty_used else ""
    print(f"  {display}{tag}")

print(f"\nFit both models for {len(rows)}/{len(models)} decision-makers.")

  Claude Opus 4.5
  Baidu Ernie 4.5 VL
  DeepSeek Chat
  Gemini 3 Pro
  Meta Llama 4 Maverick
  Mistral AI Large
  Moonshot AI Kimi K2
  OpenAI GPT 5.2
  Perplexity Sonar Pro
  Qwen 3 Max
  X-AI Grok 4
  Zhipu AI GLM 4.6
  Physician Consensus

Fit both models for 13/13 decision-makers.


## Step 2 — Marginal profiles from interaction model

**Marginal weight** for value $v$:

$$w_v^{\text{marginal}} = w_v + \sum_{v' \neq v} \gamma_{v,v'} \;\overline{\Delta_{v'}}$$

where $\overline{\Delta_{v'}}$ is the mean of $\Delta_{i,v'}$ across the 50 benchmark cases.
We then apply softmax at $T^* = 0.262$ to obtain profiles.

In [4]:
T_STAR = 0.262

profile_rows = []
for r in rows:
    model_id = r["model_id"]

    X_base, _, _ = _build_regression_data(decisions, model_id)
    mean_delta = X_base.mean(axis=0)

    marginal_w = {}
    for v_idx, v_name in enumerate(VALUE_NAMES):
        w_v = r["full_coefs"][v_name]
        correction = 0.0
        for pair_idx, (i, j) in enumerate(INTERACTION_PAIRS):
            if i == v_idx:
                correction += r["interaction_coefs"][INTERACTION_LABELS[pair_idx]] * mean_delta[j]
            elif j == v_idx:
                correction += r["interaction_coefs"][INTERACTION_LABELS[pair_idx]] * mean_delta[i]
        marginal_w[v_name] = w_v + correction

    pi_base = softmax_profile(r["base_coefs"], temperature=T_STAR)
    pi_marginal = softmax_profile(marginal_w, temperature=T_STAR)

    max_dev = max(abs(pi_base[v] - pi_marginal[v]) for v in VALUE_NAMES)

    profile_rows.append({
        "model_id": model_id,
        "display_name": r["display_name"],
        "pi_base": pi_base,
        "pi_marginal": pi_marginal,
        "max_abs_dev": max_dev,
    })

In [5]:
V_SHORT = {"autonomy": "A", "beneficence": "B", "nonmaleficence": "N", "justice": "J"}

compare_rows = []
for pr in profile_rows:
    row = {"Decision-maker": pr["display_name"]}
    for v in VALUE_NAMES:
        s = V_SHORT[v]
        row[f"\u03c0_{s} (base)"] = round(pr["pi_base"][v], 4)
        row[f"\u03c0_{s} (marg)"] = round(pr["pi_marginal"][v], 4)
    row["Max |\u0394\u03c0|"] = round(pr["max_abs_dev"], 4)
    compare_rows.append(row)

compare_df = pd.DataFrame(compare_rows).set_index("Decision-maker")
compare_df

,π_A (base),π_A (marg),π_B (base),π_B (marg),π_N (base),π_N (marg),π_J (base),π_J (marg),Max |Δπ|
Decision-maker,,,,,,,,,
Claude Opus 4.5,0.2049,0.1318,0.2183,0.2191,0.5260,0.6224,0.0507,0.0267,0.0964
Baidu Ernie 4.5 VL,0.1725,0.1832,0.2702,0.3228,0.1604,0.1650,0.3969,0.3290,0.0679
DeepSeek Chat,0.2856,0.1905,0.1581,0.1366,0.1110,0.0171,0.4453,0.6558,0.2105
Gemini 3 Pro,0.3119,0.3069,0.3549,0.3601,0.2224,0.2375,0.1109,0.0955,0.0154
Meta Llama 4 Maverick,0.1290,0.1244,0.4593,0.4374,0.1558,0.1414,0.2558,0.2968,0.0410
Mistral AI Large,0.2208,0.1388,0.3274,0.4507,0.2112,0.3938,0.2405,0.0167,0.2238
Moonshot AI Kimi K2,0.0685,0.0278,0.5854,0.7938,0.1689,0.1025,0.1772,0.0760,0.2084
OpenAI GPT 5.2,0.0607,0.0243,0.6765,0.8099,0.1211,0.0245,0.1417,0.1414,0.1334
Perplexity Sonar Pro,0.1279,0.9999,0.1610,0.0000,0.3195,0.0000,0.3915,0.0000,0.8720


## Step 3 — Rank agreement (Kendall τ)

Per-value Kendall τ between original and marginal profiles across the 12 LLMs,
plus a sensitivity check excluding the two models with degenerate interaction fits
(Qwen 3 Max, Perplexity Sonar Pro).

In [6]:
llm_rows = [pr for pr in profile_rows if pr["model_id"] != HUMAN_CONSENSUS]

HDR = "  {:<16s}  {:>8s}  {:>10s}".format("value", "τ", "p")
SEP = "  " + "─" * 38

print(f"Per-value Kendall τ: base vs marginal profiles (n={len(llm_rows)} LLMs)\n")
print(HDR)
print(SEP)
for v in VALUE_NAMES:
    base_v = [pr["pi_base"][v] for pr in llm_rows]
    marg_v = [pr["pi_marginal"][v] for pr in llm_rows]
    tau, p_tau = kendalltau(base_v, marg_v)
    print(f"  {v:<16s}  {tau:>8.4f}  {p_tau:>10.4g}")

devs = [pr["max_abs_dev"] for pr in llm_rows]
print(f"\n  Median max |Δπ|:  {np.median(devs):.4f}")
print(f"  Max    max |Δπ|:  {max(devs):.4f}")

Per-value Kendall τ: base vs marginal profiles (n=12 LLMs)

  value                    τ           p
  ──────────────────────────────────────
  autonomy            0.4848     0.03105
  beneficence         0.6061     0.00538
  nonmaleficence      0.3030      0.1969
  justice             0.5152     0.02098

  Median max |Δπ|:  0.1915
  Max    max |Δπ|:  0.9016


In [7]:
DEGENERATE = {"qwen/qwen3-max", "perplexity/sonar-pro"}
stable_rows = [pr for pr in llm_rows if pr["model_id"] not in DEGENERATE]

print(f"Sensitivity check: excluding Qwen 3 Max & Perplexity Sonar Pro (n={len(stable_rows)})\n")
print(HDR)
print(SEP)
for v in VALUE_NAMES:
    base_v = [pr["pi_base"][v] for pr in stable_rows]
    marg_v = [pr["pi_marginal"][v] for pr in stable_rows]
    tau, p_tau = kendalltau(base_v, marg_v)
    print(f"  {v:<16s}  {tau:>8.4f}  {p_tau:>10.4g}")

devs = [pr["max_abs_dev"] for pr in stable_rows]
print(f"\n  Median max |Δπ|:  {np.median(devs):.4f}")
print(f"  Max    max |Δπ|:  {max(devs):.4f}")

Sensitivity check: excluding Qwen 3 Max & Perplexity Sonar Pro (n=10)

  value                    τ           p
  ──────────────────────────────────────
  autonomy            0.8222   0.0003577
  beneficence         0.8667   0.0001152
  nonmaleficence      0.8667   0.0001152
  justice             0.6444    0.009148

  Median max |Δπ|:  0.1612
  Max    max |Δπ|:  0.2238
